In [1]:
import numpy as np
import kwant
import matplotlib.pyplot as plt

from sympy import Matrix, nsimplify
from IPython.display import display

In [2]:
def show_matrix(M):
    display(Matrix(np.asarray(M)).applyfunc(nsimplify))

In [3]:
s0 = np.eye(2, dtype=complex)

sx = np.array([
    [0, 1],
    [1, 0]
], dtype=complex)

sy = np.array([
    [0, -1j],
    [1j, 0]
], dtype=complex)

sz = np.array([
    [1, 0],
    [0, -1]
], dtype=complex)

z2 = np.zeros((2, 2), dtype=complex)

## $1\times1$ system

This has only 1 unit cell and thus the total Hamiltonian size is $16 \times 16$.

`onsite_clean` constructs a BdG matrix with onsite pairing.

In [4]:
def onsite_clean(site, mu, Delta, J):

    h = -mu * s0
    pairing = 1j * Delta * sy

    return np.block([
        [h, pairing],
        [pairing.conj().T, -h.T]
    ])

The clean electron Hamiltonian at one site is simply

$$
h=-\mu\sigma_0.
$$

The pairing matrix is

$$
\Delta i\sigma_y.
$$

The BdG onsite matrix is

$$
H_i=
\begin{pmatrix}
h & i\Delta\sigma_y\\
(i\Delta\sigma_y)^\dagger & -h^T
\end{pmatrix}.
$$

```text
0 : c_up
1 : c_down
2 : c_up^dagger
3 : c_down^dagger
```

In [5]:
mu = 0.3
Delta = 0.2
J = 0.4

H_clean = onsite_clean(None, mu=mu, Delta=Delta, J=J)
# show_matrix(H_clean)
Matrix(H_clean)

Matrix([
[-0.3,    0,    0, 0.2],
[   0, -0.3, -0.2,   0],
[   0, -0.2,  0.3,   0],
[ 0.2,    0,    0, 0.3]])

For a magnetic impurity polarized along $z$,

$$
h=-\mu\sigma_0+J\sigma_z.
$$


In [6]:
def onsite_impurity(site, mu, Delta, J):

    h = -mu * s0 + J * sz
    pairing = 1j * Delta * sy

    return np.block([
        [h, pairing],
        [pairing.conj().T, -h.T]
    ])

In [7]:
H_imp = onsite_impurity(None, mu=mu, Delta=Delta, J=J)

Matrix(H_imp)

Matrix([
[0.1,    0,    0, 0.2],
[  0, -0.7, -0.2,   0],
[  0, -0.2, -0.1,   0],
[0.2,    0,    0, 0.7]])

### Let us compare the clean and impurity sites

```python
difference = H_imp - H_clean
```

$$
H_{\rm imp}-H_{\rm clean}
=
\begin{pmatrix}
J&0&0&0\\
0&-J&0&0\\
0&0&-J&0\\
0&0&0&J
\end{pmatrix}.
$$

This is precisely

$$
J M_{\rm imp},
$$

where

```python
MIMP = np.block([
    [sz, z2],
    [z2, -sz]
])
```

In [8]:
difference = H_imp - H_clean

Matrix(difference)

Matrix([
[0.4,    0,    0,   0],
[  0, -0.4,    0,   0],
[  0,    0, -0.4,   0],
[  0,    0,    0, 0.4]])

For ordinary spin-independent electron hopping,

$$
h_{ij}=-t\sigma_0.
$$

In BdG form,

$$
T_{ij}
=
\begin{pmatrix}
h_{ij}&0\\
0&-h_{ij}^*
\end{pmatrix}.
$$

In [9]:
def hopping_bdg(t):

    he = -t * s0

    return np.block([
        [he, z2],
        [z2, -he.conj()]
    ])

In [10]:
T = hopping_bdg(1.0)

show_matrix(T)

Matrix([
[-1,  0, 0, 0],
[ 0, -1, 0, 0],
[ 0,  0, 1, 0],
[ 0,  0, 0, 1]])